# Aquarium Object Detection with YOLOv8n

**CSCI1409 Deep Learning — Bonus Project**

This notebook downloads the Aquarium dataset from Kaggle, checks YOLO annotations, creates a dataset YAML configuration, trains four YOLOv8n configurations, selects a checkpoint using validation performance, evaluates it on the held-out test split, and visualizes predictions and errors.

- **Runtime:** Google Colab with GPU enabled (Tesla T4 was used for the recorded experiments).
- **Dataset:** `slavkoprytula/aquarium-data-cots` on Kaggle.
- **Classes:** fish, jellyfish, penguin, puffin, shark, starfish, stingray.

Run the notebook from top to bottom in a fresh Colab runtime. Dataset downloads and training outputs are written under `/content/aquarium_project/`.


In [ ]:
!pip install -q kagglehub ultralytics

In [ ]:
import kagglehub

dataset_path = kagglehub.dataset_download(
    "slavkoprytula/aquarium-data-cots"
)

print("Dataset path:", dataset_path)

Inspect the structure

In [ ]:
from pathlib import Path

root = Path(dataset_path)

for path in sorted(root.iterdir()):
    print(path.name, "Directory" if path.is_dir() else "File")

In [ ]:
image_extensions = {".jpg", ".jpeg", ".png"}

images = [
    p for p in root.rglob("*")
    if p.suffix.lower() in image_extensions
]

labels = list(root.rglob("*.txt"))

print("Images:", len(images))
print("Annotation files:", len(labels))

print("\nSample images:")
for p in images[:10]:
    print(p)

Inspect the dataset structure

In [ ]:
from pathlib import Path

root = Path(dataset_path)

for path in sorted(root.rglob("*")):
    if path.is_dir() and path.name in [
        "train", "valid", "test",
        "images", "labels"
    ]:
        print(path.relative_to(root))

In [ ]:
from pathlib import Path

root = Path(dataset_path)

for split in ["train", "valid", "test"]:

    image_dir = root / "aquarium_pretrain" / split / "images"
    label_dir = root / "aquarium_pretrain" / split / "labels"

    if not image_dir.exists():
        print(f"{split}: folder not found")
        continue

    image_files = [
        p for p in image_dir.iterdir()
        if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
    ]

    label_files = list(label_dir.glob("*.txt"))

    image_stems = {p.stem for p in image_files}
    label_stems = {p.stem for p in label_files}

    print(f"\n{split.upper()}")
    print("Images:", len(image_files))
    print("Labels:", len(label_files))
    print("Missing labels:", len(image_stems - label_stems))
    print("Extra labels:", len(label_stems - image_stems))

    print("Extra label examples:", list(label_stems - image_stems)[:5])

In [ ]:
yaml_files = list(root.rglob("*.yaml"))

for file in yaml_files:
    print("\nConfiguration:", file)
    print(file.read_text())

In [ ]:
from pathlib import Path
from collections import Counter

root = Path(dataset_path) / "aquarium_pretrain"

class_names = [
    "fish",
    "jellyfish",
    "penguin",
    "puffin",
    "shark",
    "starfish",
    "stingray"
]

object_counts = Counter()
errors = []

for split in ["train", "valid", "test"]:

    label_dir = root / split / "labels"
    count = 0

    for label_file in label_dir.glob("*.txt"):

        for line_number, line in enumerate(
            label_file.read_text().splitlines(), 1
        ):

            if not line.strip():
                continue

            try:
                values = list(map(float, line.split()))

                if len(values) != 5:
                    raise ValueError("Expected 5 values")

                cls, x, y, w, h = values

                if cls != int(cls) or not 0 <= int(cls) < 7:
                    raise ValueError("Invalid class ID")

                if not (0 <= x <= 1 and 0 <= y <= 1):
                    raise ValueError("Invalid center coordinates")

                if not (0 < w <= 1 and 0 < h <= 1):
                    raise ValueError("Invalid box dimensions")

                if x - w/2 < -1e-6 or x + w/2 > 1 + 1e-6:
                    raise ValueError("Box outside image width")

                if y - h/2 < -1e-6 or y + h/2 > 1 + 1e-6:
                    raise ValueError("Box outside image height")

                object_counts[int(cls)] += 1
                count += 1

            except Exception as e:
                errors.append(
                    f"{split}/{label_file.name}, line {line_number}: {e}"
                )

    print(f"{split.upper()}: {count} valid objects")

print("\nOBJECTS PER CLASS")

for i, name in enumerate(class_names):
    print(f"{name}: {object_counts[i]}")

print("\nTotal objects:", sum(object_counts.values()))
print("Annotation errors:", len(errors))

for error in errors[:20]:
    print(error)


In [ ]:
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

random.seed(42)

image_dir = root / "train" / "images"
label_dir = root / "train" / "labels"

image_files = list(image_dir.glob("*.jpg"))
samples = random.sample(image_files, 5)

fig, axes = plt.subplots(1, 5, figsize=(22, 6))

for ax, image_path in zip(axes, samples):

    image = Image.open(image_path).convert("RGB")
    width, height = image.size

    ax.imshow(image)

    label_path = label_dir / f"{image_path.stem}.txt"

    for line in label_path.read_text().splitlines():

        if not line.strip():
            continue

        cls, x, y, w, h = map(float, line.split())

        x1 = (x - w/2) * width
        y1 = (y - h/2) * height

        rect = patches.Rectangle(
            (x1, y1),
            w * width,
            h * height,
            linewidth=2,
            edgecolor="lime",
            facecolor="none"
        )

        ax.add_patch(rect)

        ax.text(
            x1, y1,
            class_names[int(cls)],
            color="white",
            fontsize=9,
            bbox=dict(facecolor="green")
        )

    ax.axis("off")

plt.suptitle("Aquarium Dataset - Ground Truth Annotations")
plt.tight_layout()
plt.show()

### Annotation quality note

The annotation audit found **two invalid bounding-box dimension entries** in test label files. They are reported by the validation cell rather than silently modified. The recorded Ultralytics test evaluation reported 584 instances, while the annotation audit counted 582 valid test objects. This discrepancy should be investigated if the dataset annotations are revised or the experiment is rerun. Do not claim that every annotation is error-free.


Create our YOLO configuration

In [ ]:
import os
import yaml
import kagglehub
from pathlib import Path

# 1. Download or locate the dataset
dataset_path = kagglehub.dataset_download(
    "slavkoprytula/aquarium-data-cots"
)

# 2. Define dataset root
root = Path(dataset_path) / "aquarium_pretrain"

# 3. Define classes
class_names = [
    "fish",
    "jellyfish",
    "penguin",
    "puffin",
    "shark",
    "starfish",
    "stingray"
]

# 4. Verify dataset folders
for split in ["train", "valid", "test"]:
    image_dir = root / split / "images"
    label_dir = root / split / "labels"

    assert image_dir.exists(), f"Missing: {image_dir}"
    assert label_dir.exists(), f"Missing: {label_dir}"

    print(split, "images:", len(list(image_dir.glob("*.jpg"))))

# 5. Create configuration
data_config = {
    "path": str(root),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": 7,
    "names": class_names
}

# 6. Create working directory
WORK_DIR = "/content/aquarium_project"
os.makedirs(WORK_DIR, exist_ok=True)

config_path = os.path.join(WORK_DIR, "aquarium.yaml")

with open(config_path, "w") as f:
    yaml.safe_dump(data_config, f, sort_keys=False)

print("\nConfiguration created successfully!")
print(open(config_path).read())

In [ ]:
import os
from pathlib import Path

print("Colab directory exists:", os.path.exists("/content"))
print("Kaggle directory exists:", os.path.exists("/kaggle"))

print("Dataset exists:", root.exists())
print("Dataset path:", root)

print("Configuration exists:", os.path.exists(config_path))

In [ ]:
%pip install -q ultralytics

In [ ]:
from ultralytics import YOLO
import torch
import time

assert torch.cuda.is_available(), "Enable GPU in Colab!"

model = YOLO("yolov8n.pt")

start_time = time.time()

results = model.train(
    data=config_path,
    epochs=5,
    imgsz=320,
    batch=16,
    device=0,
    workers=2,
    seed=42,
    patience=5,
    project="/content/aquarium_project/runs",
    name="baseline",
    pretrained=True,
    plots=True,
    save=True,
    cache=True
)

print("Training time:", round((time.time() - start_time) / 60, 2), "minutes")

In [ ]:
from ultralytics import YOLO
import torch
import time

assert torch.cuda.is_available()

# Load the same original pretrained weights
model2 = YOLO("yolov8n.pt")

start_time = time.time()

results2 = model2.train(
    data=config_path,
    epochs=5,
    imgsz=416,          # Only intentional change
    batch=16,
    device=0,
    workers=2,
    seed=42,
    patience=5,
    project="/content/aquarium_project/runs",
    name="experiment_416",
    pretrained=True,
    plots=True,
    save=True,
    cache=False
)

experiment_time = time.time() - start_time

print("Experiment 2 time:",
      round(experiment_time / 60, 2), "minutes")

In [ ]:

from ultralytics import YOLO
import torch
import time

assert torch.cuda.is_available(), "GPU is not available"

# Start from the same original pretrained weights
model3 = YOLO("yolov8n.pt")

start_time = time.time()

results3 = model3.train(
    data=config_path,
    epochs=30,              # Increased from 5 to 30
    imgsz=416,              # Keep the best image size so far
    batch=16,
    device=0,
    workers=2,
    seed=42,
    patience=10,
    project="/content/aquarium_project/runs",
    name="experiment_30epochs",
    pretrained=True,
    plots=True,
    save=True,
    cache=False
)

training_time = time.time() - start_time

print("Training completed!")
print("Training time:", round(training_time / 60, 2), "minutes")
print("Best model:")
print("/content/aquarium_project/runs/experiment_30epochs/weights/best.pt")


In [ ]:

from ultralytics import YOLO
import torch
import time

assert torch.cuda.is_available()

model4 = YOLO("yolov8n.pt")

start_time = time.time()

results4 = model4.train(
    data=config_path,
    epochs=30,
    imgsz=512,          # Only intentional change
    batch=16,
    device=0,
    workers=2,
    seed=42,
    patience=10,
    project="/content/aquarium_project/runs",
    name="experiment_512",
    pretrained=True,
    plots=True,
    save=True,
    cache=False
)

print("Training time:",
      round((time.time() - start_time) / 60, 2),
      "minutes")


In [ ]:

from ultralytics import YOLO

model = YOLO(
    "/content/aquarium_project/runs/experiment_512/weights/best.pt"
)

thresholds = [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40]

print("Confidence | Precision | Recall | mAP50")
print("-" * 48)

for conf in thresholds:
    metrics = model.val(
        data=config_path,
        split="val",
        imgsz=512,
        batch=16,
        device=0,
        conf=conf,
        plots=False,
        verbose=False
    )

    print(
        f"{conf:>10.2f} | "
        f"{metrics.box.mp:>9.4f} | "
        f"{metrics.box.mr:>6.4f} | "
        f"{metrics.box.map50:>5.4f}"
    )


test evaluation

In [ ]:

from ultralytics import YOLO
from pathlib import Path
import matplotlib.pyplot as plt
import random

model = YOLO(
    "/content/aquarium_project/runs/experiment_512/weights/best.pt"
)

test_images = sorted(
    (Path(root) / "test" / "images").glob("*")
)

test_images = [
    p for p in test_images
    if p.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp"]
]

# Select five reproducible test images
random.seed(42)
selected_images = random.sample(test_images, min(5, len(test_images)))

predictions = model.predict(
    source=[str(p) for p in selected_images],
    imgsz=512,
    conf=0.20,
    device=0,
    save=False,
    verbose=False
)

fig, axes = plt.subplots(5, 1, figsize=(12, 24))

for ax, result, image_path in zip(axes, predictions, selected_images):
    plotted_image = result.plot()
    ax.imshow(plotted_image[:, :, ::-1])
    ax.set_title(image_path.name)
    ax.axis("off")

plt.tight_layout()
plt.savefig(
    "/content/aquarium_project/five_test_predictions.png",
    dpi=150,
    bbox_inches="tight"
)
plt.show()


In [ ]:

from PIL import Image, ImageDraw
from pathlib import Path
import matplotlib.pyplot as plt

class_names = [
    "fish", "jellyfish", "penguin", "puffin",
    "shark", "starfish", "stingray"
]

def draw_ground_truth(image_path):
    image = Image.open(image_path).convert("RGB")
    draw = ImageDraw.Draw(image)

    label_path = (
        Path(image_path).parent.parent
        / "labels" / (Path(image_path).stem + ".txt")
    )

    width, height = image.size

    if label_path.exists():
        for line in label_path.read_text().splitlines():
            values = line.split()
            if len(values) != 5:
                continue

            cls, xc, yc, bw, bh = map(float, values)
            x1 = (xc - bw / 2) * width
            y1 = (yc - bh / 2) * height
            x2 = (xc + bw / 2) * width
            y2 = (yc + bh / 2) * height

            draw.rectangle((x1, y1, x2, y2), outline="lime", width=3)
            draw.text(
                (x1, max(0, y1 - 15)),
                class_names[int(cls)],
                fill="lime"
            )

    return image

# Display three examples from the test set
failure_examples = selected_images[:3]

failure_predictions = model.predict(
    source=[str(p) for p in failure_examples],
    imgsz=512,
    conf=0.20,
    device=0,
    save=False,
    verbose=False
)

fig, axes = plt.subplots(3, 2, figsize=(16, 18))

for i, (path, prediction) in enumerate(
    zip(failure_examples, failure_predictions)
):
    gt_image = draw_ground_truth(path)
    pred_image = prediction.plot()[:, :, ::-1]

    axes[i, 0].imshow(gt_image)
    axes[i, 0].set_title("Ground truth: " + path.name)
    axes[i, 0].axis("off")

    axes[i, 1].imshow(pred_image)
    axes[i, 1].set_title("Model predictions")
    axes[i, 1].axis("off")

plt.tight_layout()
plt.savefig(
    "/content/aquarium_project/ground_truth_vs_predictions.png",
    dpi=150,
    bbox_inches="tight"
)
plt.show()


In [ ]:

from ultralytics import YOLO
import pandas as pd
import numpy as np

# Load your best trained model
model = YOLO(
    "/content/aquarium_project/runs/experiment_512/weights/best.pt"
)

# Evaluate on the held-out test dataset
results = model.val(
    data=config_path,
    split="test",
    imgsz=512,
    batch=16,
    device=0,
    conf=0.20,
    plots=True,
    verbose=True
)

# Overall metrics
print("\n========== OVERALL TEST METRICS ==========")
print(f"Precision:       {results.box.mp:.4f}")
print(f"Recall:          {results.box.mr:.4f}")
print(f"mAP@0.50:        {results.box.map50:.4f}")
print(f"mAP@0.50:0.95:   {results.box.map:.4f}")
print(f"mAP@0.75:        {results.box.map75:.4f}")

# Additional detection speed
print("\n========== INFERENCE SPEED ==========")
for key, value in results.speed.items():
    print(f"{key}: {value:.2f} ms/image")


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Use the confusion matrix already calculated
cm = np.asarray(results.confusion_matrix.matrix)

# Class names plus background
labels = class_names + ["Background"]

output_dir = "/content/aquarium_project/visualizations"
os.makedirs(output_dir, exist_ok=True)

# Plot raw confusion matrix
plt.figure(figsize=(11, 9))
sns.heatmap(
    cm,
    annot=True,
    fmt=".0f",
    cmap="Blues",
    xticklabels=labels,
    yticklabels=labels
)
plt.title("YOLOv8n - Test Confusion Matrix")
plt.xlabel("Predicted Class")
plt.ylabel("True Class")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()

plt.savefig(
    f"{output_dir}/confusion_matrix_raw.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

# Normalize each true-class row so each row sums to 1.
normalized_cm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)

plt.figure(figsize=(11, 9))
sns.heatmap(
    normalized_cm,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    xticklabels=labels,
    yticklabels=labels,
    vmin=0,
    vmax=1
)
plt.title("YOLOv8n - Normalized Test Confusion Matrix")
plt.xlabel("Predicted Class")
plt.ylabel("True Class")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()

plt.savefig(
    f"{output_dir}/confusion_matrix_normalized.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

print("Both confusion matrices generated successfully!")


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

# Use the confusion matrix from the held-out TEST evaluation.
matrix = np.asarray(results.confusion_matrix.matrix)
class_matrix = matrix[:-1, :-1]

# Matrix convention in the plotted output: rows = true classes,
# columns = predicted classes.
TP = np.diag(class_matrix)
FP = matrix[:, :-1].sum(axis=0) - TP
FN = matrix[:-1, :].sum(axis=1) - TP

precision = TP / np.maximum(TP + FP, 1)
recall = TP / np.maximum(TP + FN, 1)
f1 = 2 * precision * recall / np.maximum(precision + recall, 1e-10)

report = pd.DataFrame({
    "Class": class_names,
    "Precision_from_confusion_matrix": precision,
    "Recall_from_confusion_matrix": recall,
    "F1_from_confusion_matrix": f1
})

print(report.round(4).to_string(index=False))

output_dir = Path("/content/aquarium_project/visualizations")
output_dir.mkdir(parents=True, exist_ok=True)
report.to_csv(output_dir / "confusion_matrix_class_metrics.csv", index=False)


In [ ]:

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display, Image

# ==========================================
# 1. MODEL AND CLASS INFORMATION
# ==========================================

class_names = model.names
box = results.box

print("Classes:", class_names)

# ==========================================
# 2. CREATE PER-CLASS METRICS TABLE
# ==========================================

data = []

for i, class_id in enumerate(box.ap_class_index):
    precision = float(box.p[i])
    recall = float(box.r[i])

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall > 0 else 0
    )

    data.append({
        "Class": class_names[int(class_id)],
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1,
        "AP@0.50": float(box.ap50[i]),
        "AP@0.50:0.95": float(box.ap[i])
    })

df = pd.DataFrame(data)

print("\n========== PER-CLASS TEST METRICS ==========")
display(df.round(4))

# ==========================================
# 3. CONFUSION MATRIX
# ==========================================

cm_object = getattr(results, "confusion_matrix", None)

if cm_object is not None:
    cm = getattr(cm_object, "matrix", cm_object)
    cm = np.asarray(cm)

    labels = [class_names[i] for i in range(len(class_names))]
    labels.append("Background")

    # Raw confusion matrix
    fig, ax = plt.subplots(figsize=(11, 9))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_title("YOLOv8n - Test Confusion Matrix", fontsize=16)
    ax.set_xlabel("Predicted Class")
    ax.set_ylabel("True Class")
    ax.set_xticks(range(len(labels)))
    ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_yticklabels(labels)

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, f"{cm[i, j]:.0f}",
                    ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() * 0.5
                    else "black")

    fig.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.show()

    # Normalized confusion matrix (row percentages)
    row_sums = cm.sum(axis=1, keepdims=True)
    cm_normalized = np.divide(
        cm, row_sums,
        out=np.zeros_like(cm, dtype=float),
        where=row_sums != 0
    )

    fig, ax = plt.subplots(figsize=(11, 9))
    im = ax.imshow(cm_normalized, cmap="Blues", vmin=0, vmax=1)
    ax.set_title("Normalized Test Confusion Matrix", fontsize=16)
    ax.set_xlabel("Predicted Class")
    ax.set_ylabel("True Class")
    ax.set_xticks(range(len(labels)))
    ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_yticklabels(labels)

    for i in range(cm_normalized.shape[0]):
        for j in range(cm_normalized.shape[1]):
            ax.text(j, i, f"{cm_normalized[i, j]:.1%}",
                    ha="center", va="center",
                    color="white" if cm_normalized[i, j] > 0.5
                    else "black")

    fig.colorbar(im, ax=ax, label="Proportion")
    plt.tight_layout()
    plt.show()

else:
    print("Confusion matrix object is not exposed by this Ultralytics result.")
    print("Looking for the automatically saved confusion matrix plots...")

    possible_dirs = [
        Path(getattr(results, "save_dir", "")),
        Path("/content/aquarium_project/final_test_512"),
        Path("/content/aquarium_project/runs/detect/val")
    ]

    for folder in possible_dirs:
        if folder.is_dir():
            for filename in [
                "confusion_matrix.png",
                "confusion_matrix_normalized.png"
            ]:
                image_path = folder / filename
                if image_path.exists():
                    print(filename)
                    display(Image(filename=str(image_path)))

# ==========================================
# 4. PRECISION AND RECALL BY CLASS
# ==========================================

x = np.arange(len(df))
width = 0.35

fig, ax = plt.subplots(figsize=(11, 6))

ax.bar(x - width/2, df["Precision"], width, label="Precision")
ax.bar(x + width/2, df["Recall"], width, label="Recall")

ax.set_title("Precision and Recall by Class")
ax.set_ylabel("Score")
ax.set_xlabel("Animal Class")
ax.set_ylim(0, 1.1)
ax.set_xticks(x)
ax.set_xticklabels(df["Class"], rotation=30)
ax.legend()
ax.grid(axis="y", alpha=0.25)

plt.tight_layout()
plt.show()

# ==========================================
# 5. F1-SCORE BY CLASS
# ==========================================

fig, ax = plt.subplots(figsize=(10, 5))

ax.bar(df["Class"], df["F1-score"])
ax.set_title("F1-score by Class")
ax.set_ylabel("F1-score")
ax.set_ylim(0, 1.1)
ax.tick_params(axis="x", rotation=30)
ax.grid(axis="y", alpha=0.25)

plt.tight_layout()
plt.show()

# ==========================================
# 6. AP@0.50 BY CLASS
# ==========================================

fig, ax = plt.subplots(figsize=(10, 5))

ax.bar(df["Class"], df["AP@0.50"])
ax.set_title("Average Precision (AP@0.50) by Class")
ax.set_ylabel("AP@0.50")
ax.set_ylim(0, 1.1)
ax.tick_params(axis="x", rotation=30)
ax.grid(axis="y", alpha=0.25)

plt.tight_layout()
plt.show()

# ==========================================
# 7. AP@0.50:0.95 BY CLASS
# ==========================================

fig, ax = plt.subplots(figsize=(10, 5))

ax.bar(df["Class"], df["AP@0.50:0.95"])
ax.set_title("Average Precision (AP@0.50:0.95) by Class")
ax.set_ylabel("AP@0.50:0.95")
ax.set_ylim(0, 1.1)
ax.tick_params(axis="x", rotation=30)
ax.grid(axis="y", alpha=0.25)

plt.tight_layout()
plt.show()

# ==========================================
# 8. TRAINING CURVES
# ==========================================

training_csv = Path(
    "/content/aquarium_project/runs/experiment_512/results.csv"
)

if training_csv.exists():

    history = pd.read_csv(training_csv)
    history.columns = history.columns.str.strip()

    print("\n========== TRAINING HISTORY ==========")
    display(history.tail())

    # Plot all available loss curves
    loss_columns = [
        col for col in history.columns
        if "loss" in col.lower()
    ]

    if loss_columns:
        fig, ax = plt.subplots(figsize=(11, 6))

        for col in loss_columns:
            ax.plot(history.index + 1, history[col], label=col)

        ax.set_title("YOLOv8n Training and Validation Loss")
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Loss")
        ax.legend()
        ax.grid(alpha=0.25)

        plt.tight_layout()
        plt.show()

    # Plot validation metrics across epochs
    metric_columns = [
        col for col in history.columns
        if any(term in col.lower() for term in
               ["precision", "recall", "map50", "map50-95"])
    ]

    if metric_columns:
        fig, ax = plt.subplots(figsize=(11, 6))

        for col in metric_columns:
            ax.plot(history.index + 1, history[col], label=col)

        ax.set_title("Validation Performance Across Epochs")
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Metric Score")
        ax.set_ylim(0, 1)
        ax.legend()
        ax.grid(alpha=0.25)

        plt.tight_layout()
        plt.show()

else:
    print("Training results.csv was not found at:", training_csv)

# ==========================================
# 9. SAVE PER-CLASS METRICS
# ==========================================

output_dir = Path("/content/aquarium_project/visualizations")
output_dir.mkdir(parents=True, exist_ok=True)

df.to_csv(output_dir / "per_class_test_metrics.csv", index=False)

print("\nAll available visualizations generated.")
print("Per-class metrics saved to:", output_dir)
